# Segmentación por grupo demográfico — Reto 1 (perfiles ciudadanos de participación)

**Pregunta guía del reto:** ¿qué tipos de ciudadanía existen según su comportamiento de participación?

**Unidad de análisis (nueva):** una *celda demográfica-territorial* = **municipio × tipo de sección (urbana/mixta/rural) × grupo de edad × sexo**.
Ejemplo de celda: "mujeres de 30–39 años, secciones urbanas de Puebla (municipio X)". Hay ~60,000 celdas que cubren ~85 millones de ciudadanos de la lista nominal.

**Cómo se segmenta**
1. Cada celda tiene una trayectoria de participación en las 6 elecciones (2009–2024).
2. Se resume en 3 variables de comportamiento, **relativas al promedio nacional de cada año**: nivel, sensibilidad al ciclo presidencial/intermedio y tendencia.
3. Se agrupan las celdas con K-Means (pesadas por √ del tamaño de la celda, para que los municipios grandes pesen más sin dominar).
4. Cada perfil se **caracteriza sociodemográficamente**: composición por edad, sexo, tipo de zona y región, más su historial de participación.

**Por qué edad y sexo NO entran al clustering:** en las pruebas, al incluirlos como entradas los grupos se partían simplemente por sexo (clusters 100% hombres o 100% mujeres) y dejaban de describir comportamiento. Agrupar por comportamiento y *luego* describir quién compone cada grupo evita ese artefacto y responde directamente la pregunta guía.

**Alcance:** esta versión no usa escolaridad. Los resultados se guardan con sufijo `_demo`.


In [ ]:
# os: rutas | zipfile: leer ZIP sin descomprimir | warnings: ocultar avisos
import os, zipfile, warnings
# numpy y pandas: cálculo y tablas | matplotlib: gráficas
import numpy as np, pandas as pd, matplotlib.pyplot as plt
# TwoSlopeNorm: escala de color centrada en 1 para el mapa de calor de composición
from matplotlib.colors import TwoSlopeNorm
# chi2_contingency: para la V de Cramér | linear_sum_assignment: emparejar clusters de dos modelos
from scipy.stats import chi2_contingency
from scipy.optimize import linear_sum_assignment
# StandardScaler: estandarizar variables
from sklearn.preprocessing import StandardScaler
# KMeans: algoritmo principal | GaussianMixture: pertenencia suave (probabilidades)
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
# Métricas internas y de comparación entre agrupamientos
from sklearn.metrics import (silhouette_score, davies_bouldin_score, calinski_harabasz_score,
                             adjusted_rand_score)

# Oculta advertencias para que la salida quede limpia
warnings.filterwarnings("ignore")

# ---------------- CONFIGURACIÓN ----------------
# Carpeta donde están los ZIP de cada año
ZIP_DIR = "C:/INE/DataINE"
# Años de elección disponibles
ANIOS = [2009, 2012, 2015, 2018, 2021, 2024]
# Elecciones presidenciales e intermedias (para medir la sensibilidad al ciclo)
PRES, INTER = [2012, 2018, 2024], [2009, 2015, 2021]
# Carpeta de trabajo: aquí se guardan los resultados
CARPETA = os.getcwd()
# Semilla para reproducibilidad
SEED = 42
# Número de perfiles del modelo final (ver la comparación de k más abajo)
K = 5
# Condición del reto: entre 4 y 6 perfiles. Si K se sale de ese rango, el notebook se detiene aquí
assert 4 <= K <= 6, "El reto pide entre 4 y 6 perfiles: ajusta K"
# Rango de k que se COMPARA (3 a 8): solo sirve para justificar la elección; el modelo final respeta 4-6
K_RANGO = range(3, 9)
# Filtros de calidad de cada celda:
MIN_LN = 100      # lista nominal media mínima de la celda (evita tasas ruidosas)
MIN_D = 30        # base mínima (votaron + no votaron) en una elección para calcular su tasa
MIN_ELEC = 3      # mínimo de elecciones con tasa calculable
# Cortes de edad y sus etiquetas (7 grupos finos)
BINS = [17, 24, 29, 39, 49, 59, 69, 200]
LABS = ["18_24", "25_29", "30_39", "40_49", "50_59", "60_69", "70_mas"]
# Orden de los grupos de edad (para tablas y gráficas)
ORDEN_EDAD = {l: i for i, l in enumerate(LABS)}
# Macro-regiones (SUPUESTO PROPIO, no oficial) según la clave de estado del INE
REGIONES = {**{k: "Noroeste" for k in (2, 3, 8, 10, 25, 26)},
            **{k: "Noreste" for k in (5, 19, 28)},
            **{k: "Occidente-Bajío" for k in (1, 6, 11, 14, 16, 18, 22, 24, 32)},
            **{k: "Centro" for k in (9, 13, 15, 17, 21, 29)},
            **{k: "Sur-Sureste" for k in (4, 7, 12, 20, 23, 27, 30, 31)}}
# Nombres de las variables de comportamiento
VARS = ["nivel_rel", "ciclo_rel", "tend_rel"]
# Une la carpeta de trabajo con un nombre de archivo
ruta = lambda f: os.path.join(CARPETA, f)


## 1. Agregación desde los ZIP (municipio × tipo × edad × sexo)


In [ ]:
# Columnas numéricas que deben convertirse
NUM = ["AELEC", "EDOCVE", "MPIOCVE", "SECCION", "SEXO", "EDAD", "LN", "SV", "NV", "NS"]
# Llave de cada celda demográfica-territorial
LLAVE = ["EDOCVE", "MPIOCVE", "TIPOSEC", "GEDAD", "SEXO"]

def agregar_anio(anio):
    # Lee los 32 CSV de un año y devuelve (tabla agregada, catálogo de nombres de municipio)
    partes, nombres = [], []
    # Abre el ZIP de ese año
    with zipfile.ZipFile(os.path.join(ZIP_DIR, f"DatosAbiertos_DECEyEC_ConteosCensales{anio}.zip")) as z:
        # Recorre solo los archivos CSV
        for nombre in z.namelist():
            if not nombre.lower().endswith(".csv"):
                continue
            # Lee el CSV directamente desde el ZIP
            d = pd.read_csv(z.open(nombre), low_memory=False, encoding_errors="replace")
            # Normaliza nombres de columna
            d.columns = d.columns.str.strip().str.upper()
            # Convierte a número; lo no numérico queda NaN
            for c in NUM:
                d[c] = pd.to_numeric(d[c], errors="coerce")
            # Quita filas sin llaves básicas y deja solo mayores de edad
            d = d.dropna(subset=["AELEC", "EDOCVE", "MPIOCVE", "SECCION", "SEXO", "EDAD"])
            d = d[d.EDAD >= 18]
            # NS vacío = 0; TIPOSEC vacío = "N" (para que groupby no descarte esas filas)
            d["NS"] = d["NS"].fillna(0)
            d["TIPOSEC"] = d["TIPOSEC"].fillna("N").astype(str).str.strip().str.upper().str[0]
            # Grupo de edad fino (7 grupos)
            d["GEDAD"] = pd.cut(d.EDAD, bins=BINS, labels=LABS).astype(str)
            # Catálogo: nombre de estado y municipio por (estado, municipio)
            nombres.append(d[["EDOCVE", "MPIOCVE", "EDONOM", "MPIONOM"]]
                           .drop_duplicates(["EDOCVE", "MPIOCVE"]).assign(AELEC=anio))
            # Suma conteos por celda y elección
            partes.append(d.groupby(["AELEC"] + LLAVE, as_index=False)[["LN", "SV", "NV", "NS"]].sum())
    # Junta los archivos del año (inocuo si una celda aparece en varios)
    agg = pd.concat(partes).groupby(["AELEC"] + LLAVE, as_index=False).sum()
    return agg, pd.concat(nombres)

# Si ya se calculó antes, reutiliza el resultado (la primera vez tarda ~2 min)
if os.path.exists(ruta("agregado_demo.pkl")):
    agg = pd.read_pickle(ruta("agregado_demo.pkl")); nom_mun = pd.read_pickle(ruta("nombres_mun_demo.pkl"))
else:
    res = [agregar_anio(a) for a in ANIOS]
    # Tabla agregada de los 6 años
    agg = pd.concat([r[0] for r in res], ignore_index=True)
    # Catálogo de nombres: conserva el más reciente de cada municipio
    nom_mun = (pd.concat([r[1] for r in res]).sort_values("AELEC")
               .drop_duplicates(["EDOCVE", "MPIOCVE"], keep="last").drop(columns="AELEC"))
    # Guarda en pickle (evita problemas de pyarrow)
    agg.to_pickle(ruta("agregado_demo.pkl")); nom_mun.to_pickle(ruta("nombres_mun_demo.pkl"))

# D = base de participación conocida (excluye NS)
agg["D"] = agg.SV + agg.NV
# Participación nacional de cada año (referencia para la participación relativa)
nat = agg.groupby("AELEC").apply(lambda d: d.SV.sum() / d.D.sum())
print(agg.shape, "| participación nacional (%):", (nat * 100).round(1).to_dict())


## 2. Variables por celda


In [ ]:
# Pone la llave + año como índice para poder 'desplegar' los años en columnas
a = agg.set_index(LLAVE + ["AELEC"])
# Tasa de participación por celda y elección (NaN si la base es menor a MIN_D)
p = (a.SV / a.D.where(a.D >= MIN_D)).unstack("AELEC")
# Lista nominal por celda y elección
ln = a.LN.unstack("AELEC")
# Participación RELATIVA: tasa de la celda menos la tasa nacional de ese año
rel = p.sub(nat, axis=1)

# DataFrame de celdas con las variables de comportamiento
U = pd.DataFrame(index=p.index)
# nivel_rel: cuánto participa la celda por encima/debajo del país, en promedio
U["nivel_rel"] = rel.mean(axis=1)
# ciclo_rel: cuánto MÁS participa en presidenciales que en intermedias, comparado con el país
U["ciclo_rel"] = rel[PRES].mean(axis=1) - rel[INTER].mean(axis=1)
# tend_rel: cambio de la participación relativa entre su primera y última elección observadas
U["tend_rel"] = rel.ffill(axis=1).iloc[:, -1] - rel.bfill(axis=1).iloc[:, 0]
# Número de elecciones con tasa calculable y lista nominal media
U["n_elec"] = p.notna().sum(axis=1)
U["LN_media"] = ln.mean(axis=1)
# Participación media y por elección (para describir los perfiles)
U["part_media"] = p.mean(axis=1)
for y in ANIOS:
    U[f"part_{y}"] = p[y]
# Vuelve las llaves columnas normales
U = U.reset_index()

# Filtro de calidad: suficientes elecciones y tamaño mínimo
n_antes = len(U)
U = U[(U.n_elec >= MIN_ELEC) & (U.LN_media >= MIN_LN)].copy()
# Rellena huecos de las 3 variables con la mediana (pocas celdas)
U[VARS] = U[VARS].fillna(U[VARS].median())
# Variables descriptivas: orden de edad, macro-región e ID de municipio
U["edad_ord"] = U.GEDAD.map(ORDEN_EDAD)
U["region"] = U.EDOCVE.map(REGIONES)
U["mun_id"] = U.EDOCVE.astype(str) + "_" + U.MPIOCVE.astype(str)
U = U.reset_index(drop=True)
print(f"Celdas: {n_antes:,} -> {len(U):,} tras el filtro | municipios: {U.mun_id.nunique():,}")
print(f"Lista nominal media cubierta: {U.LN_media.sum()/1e6:.1f} millones")
print("\nComportamiento medio por grupo de edad (puntos porcentuales vs. país, ponderado):")
print(U.groupby("GEDAD").apply(lambda d: pd.Series({c: np.average(d[c], weights=d.LN_media) * 100 for c in VARS})).round(1))


## 3. Escalado y pesos


In [ ]:
# Winsorización: recorta extremos al percentil 1 y 99 de cada variable
Xw = U[VARS].clip(U[VARS].quantile(0.01), U[VARS].quantile(0.99), axis=1)
# Estandariza (media 0, desviación 1): las 3 variables pesan lo mismo
Z = StandardScaler().fit_transform(Xw)
# Peso de cada celda = raíz de su tamaño: las celdas grandes pesan más, pero sin que una sola domine
peso = np.sqrt(U.LN_media.values)
print(Z.shape)


## 4. Comparación de k


In [ ]:
# Tabla de resultados
res = []
for k in K_RANGO:
    # K-Means pesado por tamaño de celda
    km_k = KMeans(k, n_init=10, random_state=SEED).fit(Z, sample_weight=peso)
    # Estabilidad: 4 submuestras del 80%; ARI entre el modelo completo y el de la submuestra
    estab = []
    for b in range(4):
        s = np.random.RandomState(b).choice(len(Z), int(0.8 * len(Z)), replace=False)
        lb = KMeans(k, n_init=3, random_state=b).fit(Z[s], sample_weight=peso[s]).labels_
        estab.append(adjusted_rand_score(km_k.labels_[s], lb))
    res.append(dict(k=k,
                    sil=silhouette_score(Z, km_k.labels_, sample_size=10000, random_state=SEED),
                    dbi=davies_bouldin_score(Z, km_k.labels_),
                    CH=calinski_harabasz_score(Z, km_k.labels_),
                    estabilidad=np.mean(estab)))
print(pd.DataFrame(res).round(3).to_string(index=False))


## 5. Modelo final: K-Means (perfil principal) + GMM (pertenencia suave)


In [ ]:
# ---- K-Means final ----
km = KMeans(K, n_init=50, random_state=SEED).fit(Z, sample_weight=peso)
# Reordena los clusters de menor a mayor nivel de participación relativa (0 = más baja)
niv = U.groupby(km.labels_).apply(lambda d: np.average(d.nivel_rel, weights=d.LN_media))
orden = niv.sort_values().index
mapa_orden = {viejo: nuevo for nuevo, viejo in enumerate(orden)}
U["cluster"] = [mapa_orden[l] for l in km.labels_]

# ---- GMM con covarianza completa: probabilidad de pertenencia a cada componente ----
# (sklearn no admite pesos en GMM, por eso se ajusta sin ponderar; solo sirve para la pertenencia suave)
gm = GaussianMixture(K, covariance_type="full", n_init=3, random_state=SEED).fit(Z)
prob = gm.predict_proba(Z)
# Empareja cada componente del GMM con el cluster de K-Means que más se le parece (algoritmo húngaro)
tabla = pd.crosstab(gm.predict(Z), U["cluster"].values).reindex(index=range(K), columns=range(K), fill_value=0).values
fila, col = linear_sum_assignment(-tabla)
prob_k = np.zeros_like(prob)
for f, c in zip(fila, col):
    prob_k[:, c] = prob[:, f]                    # columna c = probabilidad del perfil c
U["prob_max"] = prob_k.max(axis=1)
U["ambiguo"] = U.prob_max < 0.6

# ---- Diagnóstico ----
print("ARI K-Means vs GMM:", round(adjusted_rand_score(U.cluster, prob_k.argmax(axis=1)), 3))
print(f"Celdas con pertenencia máxima < 0.6 (frontera difusa): {100*(U.prob_max<0.6).mean():.1f}%")
print(f"Celdas con pertenencia máxima >= 0.8 (núcleo claro):   {100*(U.prob_max>=0.8).mean():.1f}%")

# ---- ¿Qué tan sociodemográficos son los perfiles? V de Cramér contra cada variable descriptiva ----
def cramer_v(x, y):
    ct = pd.crosstab(x, y); c2 = chi2_contingency(ct)[0]
    return np.sqrt(c2 / ct.values.sum() / (min(ct.shape) - 1))
print("\nV de Cramér entre el perfil y:")
for nombre, colu in (("grupo de edad", "GEDAD"), ("sexo", "SEXO"), ("tipo de sección", "TIPOSEC"),
                     ("macro-región", "region"), ("estado", "EDOCVE")):
    print(f"  {nombre:16s}: {cramer_v(U.cluster.values, U[colu].values):.3f}")


## 6. Perfilado sociodemográfico


In [ ]:
# Nombres definitivos de los perfiles (rellénalo después de leer las tablas y gráficas).
# Aquí van los nombres finales usados en el informe. Si lo dejas vacío se usan nombres sugeridos automáticos.
NOMBRES = {0: "Jóvenes, participación baja y constante",
           1: "Jóvenes, participación baja que pierde terreno",
           2: "Ciudadanía en ascenso, sensible a las presidenciales",
           3: "Adultos, participación alta y estable",
           4: "Adultos, participación alta y plana (rural y mixta)"}   # válidos para K=5 y semilla 42; ajusta si cambias K o la semilla

# Promedio ponderado por lista nominal, ignorando NaN
def prom_pond(d, c):
    m = d[c].notna()
    return np.average(d.loc[m, c], weights=d.loc[m, "LN_media"])

# Composición porcentual (% de lista nominal) de cada perfil según una variable descriptiva
def composicion(col):
    t = U.pivot_table(index="cluster", columns=col, values="LN_media", aggfunc="sum").fillna(0)
    return t.div(t.sum(axis=1), axis=0) * 100

# ---- Tabla de comportamiento por perfil ----
perfil = pd.DataFrame({"celdas": U.groupby("cluster").size(),
                       "pct_lista_nominal": U.groupby("cluster").LN_media.sum() / U.LN_media.sum() * 100})
# Las 3 variables de comportamiento, en puntos porcentuales vs. el país
for c in VARS:
    perfil[c + "_pp"] = U.groupby("cluster").apply(lambda d, c=c: prom_pond(d, c) * 100)
# Participación absoluta media y por elección
perfil["part_media_pct"] = U.groupby("cluster").apply(lambda d: prom_pond(d, "part_media") * 100)
for y in ANIOS:
    perfil[f"part_{y}_pct"] = U.groupby("cluster").apply(lambda d, y=y: prom_pond(d, f"part_{y}") * 100)

# ---- Composición sociodemográfica ----
edad = composicion("GEDAD")[LABS]                       # % por grupo de edad
sexo = composicion("SEXO").rename(columns={0: "hombres", 1: "mujeres"})
tipo = composicion("TIPOSEC")[["U", "M", "R"]].rename(columns={"U": "urbana", "M": "mixta", "R": "rural"})
region = composicion("region")
# Referencia: composición de TODA la población analizada (para ver sobre/sub-representación)
def ref(col):
    t = U.groupby(col).LN_media.sum(); return t / t.sum() * 100

# ---- Nombre sugerido automático a partir de los datos ----
def nombre_sugerido(i):
    pr, e, s, t, r = perfil.loc[i], edad.loc[i], sexo.loc[i], tipo.loc[i], region.loc[i]
    # Edad: jóvenes si 18-39 >= 60%; adultos 50+ si 50+ >= 50%; adultos 40+ si 40+ >= 70%
    if e[["18_24", "25_29", "30_39"]].sum() >= 60: ed = "Jóvenes 18-39"
    elif e[["50_59", "60_69", "70_mas"]].sum() >= 50: ed = "Adultos 50+"
    elif e[["40_49", "50_59", "60_69", "70_mas"]].sum() >= 70: ed = "Adultos 40+"
    else: ed = "Todas las edades"
    # Nivel de participación relativo al país
    nv = "baja" if pr.nivel_rel_pp <= -5 else ("alta" if pr.nivel_rel_pp >= 5 else "media")
    # Dinámica RELATIVA al país: 'gana terreno' = su participación creció más que la nacional;
    # 'pierde terreno' = creció menos (en términos absolutos puede estar plana)
    dn = "gana terreno" if pr.tend_rel_pp >= 5 else ("pierde terreno" if pr.tend_rel_pp <= -5 else "estable")
    # Rasgos extra: sexo dominante, peso rural, región concentrada
    extra = []
    if s["mujeres"] >= 62: extra.append("mayoría mujeres")
    if s["hombres"] >= 58: extra.append("mayoría hombres")
    if t["rural"] >= 30: extra.append("peso rural")
    if r.max() >= 40: extra.append(f"región {r.idxmax()}")
    return f"{ed} · participación {nv}, {dn}" + (f" ({', '.join(extra)})" if extra else "")
perfil["nombre"] = [NOMBRES.get(i, nombre_sugerido(i)) for i in perfil.index]
nombre = perfil["nombre"]

# ---- Impresión ----
pd.set_option("display.width", 250); pd.set_option("display.max_columns", None)
print("COMPORTAMIENTO (pp = puntos porcentuales vs. promedio nacional del año)")
print(perfil.drop(columns="nombre").round(1).to_string())
print("\nEDAD (% de la lista nominal del perfil)");   print(edad.round(0).to_string())
print("\nSEXO (%)");                                     print(sexo.round(0).to_string())
print("\nTIPO DE SECCIÓN (%)");                          print(tipo.round(0).to_string())
print("\nMACRO-REGIÓN (%)");                             print(region.round(0).to_string())
print("\nREFERENCIA (toda la población analizada, %):")
print("  edad:", ref("GEDAD").reindex(LABS).round(0).to_dict()); print("  sexo:", ref("SEXO").round(0).to_dict())
print("  tipo:", ref("TIPOSEC").round(0).to_dict());              print("  región:", ref("region").round(0).to_dict())
print("\nNOMBRES SUGERIDOS:"); print(nombre.to_string())


## 7. Gráficas


In [ ]:
# a) Trayectoria de participación RELATIVA al promedio nacional, por perfil
# Promedio ponderado por perfil y elección de (tasa de la celda - tasa nacional)
traj = pd.DataFrame({y: [prom_pond(U[U.cluster == i], f"part_{y}") - nat[y] for i in perfil.index] for y in ANIOS},
                    index=perfil.index) * 100
fig, ax = plt.subplots(figsize=(9, 6))
for i in traj.index:
    ax.plot(ANIOS, traj.loc[i], marker="o", label=f"{i}: {nombre[i]}")
ax.axhline(0, color="gray", lw=0.8)
ax.set_title("Participación relativa al promedio nacional, por perfil"); ax.set_ylabel("Puntos porcentuales vs. país")
ax.set_xticks(ANIOS); ax.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.1))
plt.tight_layout(); plt.savefig(ruta("demo_trayectoria_relativa.png"), dpi=200); plt.show()

# b) Composición por edad, sexo, tipo de sección y región (4 paneles)
fig, axs = plt.subplots(2, 2, figsize=(13, 8))
etq = [f"Perfil {i}" for i in perfil.index]
edad.set_axis(etq).plot(kind="bar", stacked=True, ax=axs[0, 0], colormap="viridis", width=0.8)
axs[0, 0].set_title("Edad (% de la lista nominal del perfil)")
sexo.set_axis(etq).plot(kind="bar", stacked=True, ax=axs[0, 1], color=["#4c78a8", "#e45756"], width=0.8)
axs[0, 1].set_title("Sexo (%)")
tipo.set_axis(etq).plot(kind="bar", stacked=True, ax=axs[1, 0], color=["#72b7b2", "#f2cf5b", "#b07aa1"], width=0.8)
axs[1, 0].set_title("Tipo de sección (%)")
region.set_axis(etq).plot(kind="bar", stacked=True, ax=axs[1, 1], colormap="Set2", width=0.8)
axs[1, 1].set_title("Macro-región (%)")
for ax in axs.ravel():
    ax.tick_params(axis="x", rotation=0); ax.legend(fontsize=7, loc="upper right", bbox_to_anchor=(1.18, 1))
plt.tight_layout(); plt.savefig(ruta("demo_composicion.png"), dpi=200); plt.show()

# c) Mapa de calor del ÍNDICE de composición: % en el perfil / % en toda la población (1 = igual al promedio)
bloques = [(edad, ref("GEDAD").reindex(LABS)), (sexo, ref("SEXO").rename({0: "hombres", 1: "mujeres"})),
           (tipo, ref("TIPOSEC").rename({"U": "urbana", "M": "mixta", "R": "rural"})), (region, ref("region"))]
idx = pd.concat([(b / r) for b, r in bloques], axis=1)
fig, ax = plt.subplots(figsize=(11, 4.5))
im = ax.imshow(idx.values, cmap="RdBu_r", norm=TwoSlopeNorm(vmin=0.3, vcenter=1, vmax=1.7), aspect="auto")
ax.set_xticks(range(idx.shape[1])); ax.set_xticklabels(idx.columns, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(idx.shape[0])); ax.set_yticklabels([f"Perfil {i}" for i in idx.index])
plt.colorbar(im, label="Índice (1 = promedio; >1 sobre-representado)")
ax.set_title("¿Qué grupos están sobre-representados en cada perfil?")
plt.tight_layout(); plt.savefig(ruta("demo_indice_composicion.png"), dpi=200); plt.show()


## 8. Exportación


In [ ]:
# Tabla de celdas con perfil y pertenencia suave
celdas = U.merge(nom_mun, on=["EDOCVE", "MPIOCVE"], how="left")
for i in range(K):
    celdas[f"prob_perfil_{i}"] = prob_k[:, i]
celdas["perfil_nombre"] = celdas.cluster.map(nombre)
celdas.to_csv(ruta("celdas_con_perfil_demo.csv"), index=False)

# Resumen de perfiles y composiciones
perfil.to_csv(ruta("perfiles_resumen_demo.csv"))
pd.concat({"edad": edad, "sexo": sexo, "tipo": tipo, "region": region}, axis=1).to_csv(ruta("perfiles_composicion_demo.csv"))

# Distribución de perfiles por municipio (% de la lista nominal de cada municipio en cada perfil)
mm = U.pivot_table(index="mun_id", columns="cluster", values="LN_media", aggfunc="sum").fillna(0)
mm_pct = mm.div(mm.sum(axis=1), axis=0) * 100
mm_pct.columns = [f"pct_perfil_{c}" for c in mm_pct.columns]
mm_pct["perfil_dominante"] = mm.idxmax(axis=1)
mm_pct["lista_nominal"] = mm.sum(axis=1)
# Agrega nombres de estado y municipio
nm = nom_mun.assign(mun_id=nom_mun.EDOCVE.astype(str) + "_" + nom_mun.MPIOCVE.astype(str)).set_index("mun_id")
mm_pct = mm_pct.join(nm[["EDONOM", "MPIONOM"]])
mm_pct.round(1).to_csv(ruta("municipios_perfiles_demo.csv"))
print("Listo: celdas_con_perfil_demo.csv, perfiles_resumen_demo.csv, perfiles_composicion_demo.csv, municipios_perfiles_demo.csv y 3 gráficas demo_*.png")
print("Perfil dominante por municipio (n.º de municipios):", mm_pct.perfil_dominante.value_counts().sort_index().to_dict())


## 9. Cómo leer los resultados y límites

- **Los perfiles describen grupos de ciudadanos (edad × sexo × zona), no personas individuales.** Las bases son agregadas por sección.
- **La tendencia no es de una cohorte.** El grupo "18–24" de 2009 y el de 2024 son personas distintas; la tendencia mide cómo se comporta ese *grupo demográfico* a lo largo del tiempo.
- **La tendencia es relativa al país.** "Pierde terreno" significa que la participación del grupo creció menos que la nacional; en términos absolutos puede estar plana (es el caso de los perfiles 1 y 4). "Gana terreno" significa lo contrario.
- **Número de perfiles:** el reto pide de 4 a 6 y el modelo final usa K=5 (el código lo verifica con un `assert`). La comparación de k llega hasta 8 solo para justificar la elección.
- **Las fronteras son difusas.** Usa `prob_max` y `ambiguo` en `celdas_con_perfil_demo.csv`: una celda ambigua se parece a dos perfiles a la vez y conviene tratarla con un mensaje mixto.
- **Las macro-regiones son un supuesto propio** (`REGIONES`); cámbialas si el reto o el INE usan otra regionalización.
- **Sin escolaridad:** esta versión no la incluye por indicación; si más adelante se agrega, entra como variable descriptiva.
- **Filtros:** se descartan celdas con lista nominal media < 100 o con menos de 3 elecciones calculables (ver `MIN_LN`, `MIN_ELEC`).
